In [139]:
import sys
import numpy as np
import importlib

sys.path.append('../src')
import policies 
import bbDebiasing2
import wbDebiasing


## Black Box Algorithm Tests

Sanity check: if all my initial policies have single LS, should end up w predictor of mean.

In [58]:
importlib.reload(bbDebiasing2)
curr_preds = np.zeros([3,2])
my_policy = policies.Simplex(2)
other_policies = [np.array([[1,0],[1,0],[1,0]])] #debiasing wrt these LS will just give you average predictor
train_ys = np.array([[1,0],[2,-1],[3,5]])
tolerance=0.1

bbModeltest = bbDebiasing2.bbModel(my_policy, other_policies,train_ys, curr_preds, tolerance)
print(bbModeltest.debias()==np.tile(train_ys.mean(axis=0), (len(train_ys),1)))
bbModeltest.predict(curr_preds, other_policies)==np.tile(train_ys.mean(axis=0), (len(train_ys),1))

[[ True  True]
 [ True  True]
 [ True  True]]


array([[ True,  True],
       [ True,  True],
       [ True,  True]])

Sanity check: if initial policies have $n$ disjoint level sets, should converge towards exact labels. Note: they don't go exactly to this because the maximal level set in each round often is the one where the policy is 0 in multiple coordinates, so do end up iterating through LSs over many rounds until tolerance constraint is met. 

In [57]:
importlib.reload(bbDebiasing2)
curr_preds = np.zeros([3,2])
my_policy = policies.Simplex(2)
other_policies = [np.array([[1,0],[0,1],[0,0]])] #debiasing wrt these LS will just give you average predictor
train_y = np.array([[1,0],[2,-1],[3,5]])
tolerance=0.01

bbModeltest2 = bbDebiasing2.bbModel(my_policy, other_policies,train_ys,curr_preds, tolerance)
print(bbModeltest2.debias())
bbModeltest2.predict(curr_preds, other_policies)==bbModeltest2.debias()

[[ 1.0078125   0.01953125]
 [ 2.         -1.        ]
 [ 2.9921875   4.98046875]]


array([[ True,  True],
       [ True,  True],
       [ True,  True]])

## Whitebox Algorithm Tests

In [774]:
k,d,n = 2,3,5

all_policies = [policies.Simplex(d) for i in range(k)]
np.random.seed(42)
train_ys = np.random.binomial(1,0.5,(n,d)).astype(np.float64)
preds_by_models = np.random.binomial(1,0.5,(k,n,d)).astype(np.float64)
tolerance = 0.1

In [775]:
importlib.reload(wbDebiasing)
wbModel = wbDebiasing.wbModel(all_policies, train_ys, preds_by_models, tolerance)
out = wbModel.debias()

In [727]:
np.array(wbModel.mses_by_round)[:,1]

array([[0.6       , 0.8       , 0.6       ],
       [0.53333333, 0.8       , 0.33333333],
       [0.53333333, 0.8       , 0.33333333],
       [0.48888889, 0.4       , 0.05555556],
       [0.48888889, 0.4       , 0.05555556],
       [0.48148148, 0.33333333, 0.00925926],
       [0.33209877, 0.28888889, 0.00895062],
       [0.23251029, 0.25925926, 0.00874486],
       [0.23251029, 0.25925926, 0.00874486],
       [0.23251029, 0.05925926, 0.00874486],
       [0.16611797, 0.03950617, 0.00860768],
       [0.12185642, 0.02633745, 0.00851623],
       [0.12185642, 0.02633745, 0.00851623],
       [0.12185642, 0.02633745, 0.00851623],
       [0.09234873, 0.0175583 , 0.00845527],
       [0.07267693, 0.01170553, 0.00841462],
       [0.07267693, 0.01170553, 0.00841462]])

In [738]:
# note: in order to be runable, have to store preds
# oos_out, oos_rounds = wbModel.predict(preds_by_models)
# for i in range(len(oos_rounds)):
#     if (oos_rounds[i]!=wbModel.preds_by_rounds[i]).any():
#         print(i)

In [779]:
ensemble_policy, ensemble_model, expected_self_eval = wbModel.ensemble(out)
expected_self_eval

np.float64(1.0483539094650207)

In [781]:
wbModel.calc_ensemble_return(ensemble_policy, train_ys)

(np.float64(1.0), array([1., 1., 1., 1., 1.]))

In [783]:
train_ys

array([[0., 1., 1.],
       [1., 0., 0.],
       [0., 1., 1.],
       [1., 0., 1.],
       [1., 0., 0.]])

In [784]:
ensemble_policy

array([[0., 0., 1.],
       [1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.],
       [1., 0., 0.]])